# Домашнє завдання 3. Аналіз даних у PySpark

In [ ]:
!pip install -q pyspark

In [ ]:
import os

FILES = ["users.csv", "purchases.csv", "products.csv"]
os.makedirs("data", exist_ok=True)

if not all(os.path.exists(f"data/{name}") for name in FILES):
    from google.colab import files

    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(f"data/{name}", "wb") as f:
            f.write(content)

print(os.listdir("data"))

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("goit-de-hw-03").getOrCreate()

## 1. Завантаження CSV-файлів у DataFrame

In [ ]:
users_df = spark.read.csv("data/users.csv", header=True, inferSchema=True)
purchases_df = spark.read.csv("data/purchases.csv", header=True, inferSchema=True)
products_df = spark.read.csv("data/products.csv", header=True, inferSchema=True)

print("users:", users_df.count())
users_df.show(5)

print("purchases:", purchases_df.count())
purchases_df.show(5)

print("products:", products_df.count())
products_df.show(5)

## 2. Очищення даних: видалення рядків з пропущеними значеннями

In [ ]:
users_df = users_df.dropna()
purchases_df = purchases_df.dropna()
products_df = products_df.dropna()

print("users:", users_df.count())
users_df.show(5)

print("purchases:", purchases_df.count())
purchases_df.show(5)

print("products:", products_df.count())
products_df.show(5)

## 3. Загальна сума покупок за кожною категорією продуктів

In [ ]:
purchases_products_df = purchases_df.join(products_df, on="product_id", how="inner")

total_by_category_df = (
    purchases_products_df
    .groupBy("category")
    .agg(F.round(F.sum(F.col("quantity") * F.col("price")), 2).alias("total_amount"))
    .orderBy(F.col("total_amount").desc())
)

total_by_category_df.show()

## 4. Сума покупок за кожною категорією для віку від 18 до 25 включно

In [ ]:
young_df = (
    purchases_products_df
    .join(users_df, on="user_id", how="inner")
    .filter(F.col("age").between(18, 25))
)

young_by_category_df = (
    young_df
    .groupBy("category")
    .agg(F.round(F.sum(F.col("quantity") * F.col("price")), 2).alias("total_amount"))
    .orderBy(F.col("total_amount").desc())
)

young_by_category_df.show()

## 5. Частка покупок за кожною категорією від сумарних витрат для віку від 18 до 25

In [ ]:
young_total = young_by_category_df.agg(F.sum("total_amount")).collect()[0][0]
print("Сумарні витрати 18-25:", young_total)

young_share_df = (
    young_by_category_df
    .withColumn("percentage", F.round(F.col("total_amount") / young_total * 100, 2))
    .orderBy(F.col("percentage").desc())
)

young_share_df.show()

## 6. Топ-3 категорії з найвищим відсотком витрат для віку від 18 до 25

In [ ]:
top3_df = young_share_df.orderBy(F.col("percentage").desc()).limit(3)

top3_df.show()